# Router VSA K49 Finetune-Only

This notebook checks tokenizer/generation debug fixes, prepares cached features if needed, then trains `stage=finetune` without running align.

In [ ]:
GITHUB_REPO_URL = "https://github.com/huyvanzzz/Group_HND_car.git"
GITHUB_BRANCH = "huy"
USE_SAFE_CONFIG = "0"

# Optional: restore a previously uploaded cache repo. Leave empty to prepare features locally.
CACHE_HF_REPO_ID = ""
CACHE_HF_REPO_TYPE = "dataset"
CACHE_HF_SUBDIR = "cache"

# Optional: resume same-stage finetune from Hugging Face.
FINETUNE_CHECKPOINT_HF_REPO_ID = ""
FINETUNE_CHECKPOINT_HF_REPO_TYPE = "model"
FINETUNE_CHECKPOINT_FILENAME = "finetune_latest.pt"

import os, json
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["USE_SAFE_CONFIG"] = USE_SAFE_CONFIG
os.environ["CACHE_HF_REPO_ID"] = CACHE_HF_REPO_ID
os.environ["CACHE_HF_REPO_TYPE"] = CACHE_HF_REPO_TYPE
os.environ["CACHE_HF_SUBDIR"] = CACHE_HF_SUBDIR
os.environ["FINETUNE_CHECKPOINT_HF_REPO_ID"] = FINETUNE_CHECKPOINT_HF_REPO_ID
os.environ["FINETUNE_CHECKPOINT_HF_REPO_TYPE"] = FINETUNE_CHECKPOINT_HF_REPO_TYPE
os.environ["FINETUNE_CHECKPOINT_FILENAME"] = FINETUNE_CHECKPOINT_FILENAME
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass
print(json.dumps({
    "use_safe_config": USE_SAFE_CONFIG,
    "cache_hf_repo": CACHE_HF_REPO_ID,
    "finetune_checkpoint_hf_repo": FINETUNE_CHECKPOINT_HF_REPO_ID,
    "hf_token_present": bool(os.environ.get("HF_TOKEN")),
}, indent=2))

In [ ]:
%%bash
set -e
cd /kaggle/working
if [ ! -d Efficient_VLM_For_Autonomous_Driving ]; then
  git clone "$GITHUB_REPO_URL" Efficient_VLM_For_Autonomous_Driving
fi
cd Efficient_VLM_For_Autonomous_Driving
git fetch origin "$GITHUB_BRANCH"
git checkout "$GITHUB_BRANCH"
git pull --ff-only origin "$GITHUB_BRANCH"
python -m pip install -q -e . accelerate pycocoevalcap
python -m efficient_vlm_ad --help | head -n 40

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
if [ "$USE_SAFE_CONFIG" = "1" ]; then
  BASE_CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49_safe.yaml
  BASE_PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49_safe
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49_safe_finetune_only
else
  BASE_CONFIG=configs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49.yaml
  BASE_PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49
  PROFILE=outputs/repvit_t5_efficient_mini_kaggle_2gpu_router_vsa_k49_finetune_only
fi
test -f "$BASE_CONFIG"
RUNTIME_CONFIG=/tmp/repvit_t5_router_vsa_k49_finetune_only.yaml
export BASE_CONFIG BASE_PROFILE PROFILE RUNTIME_CONFIG
python - <<'PY'
import os
from pathlib import Path
import yaml
base = Path(os.environ['BASE_CONFIG'])
cfg = yaml.safe_load(base.read_text())
cfg['project']['output_dir'] = os.environ['PROFILE']
cfg['cache']['dir'] = os.environ['BASE_PROFILE'] + '/cache'
Path(os.environ['RUNTIME_CONFIG']).write_text(yaml.safe_dump(cfg, sort_keys=False))
print('runtime_config', os.environ['RUNTIME_CONFIG'])
print('output_dir', cfg['project']['output_dir'])
print('cache_dir', cfg['cache']['dir'])
print('generation', cfg.get('generation'))
PY
cat > /kaggle/working/router_finetune_only_env.sh <<EOF
BASE_CONFIG=$BASE_CONFIG
BASE_PROFILE=$BASE_PROFILE
PROFILE=$PROFILE
RUNTIME_CONFIG=$RUNTIME_CONFIG
EOF
cat /kaggle/working/router_finetune_only_env.sh

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
python - <<'PY'
import torch
from efficient_vlm_ad.config import load_config
from efficient_vlm_ad.modeling.factory import build_text_and_tokenizer
from efficient_vlm_ad.tokenization import decode_generated_predictions
cfg = load_config('/tmp/repvit_t5_router_vsa_k49_finetune_only.yaml')
model, tok = build_text_and_tokenizer(cfg)
lt_id = tok.convert_tokens_to_ids('<')
gt_id = tok.encode('>', add_special_tokens=False)[0]
emb = model.get_input_embeddings().weight.detach().cpu()
print('< id', lt_id, '> id', gt_id)
print('< norm', float(emb[lt_id].norm()), '> norm', float(emb[gt_id].norm()), 'same', bool(torch.allclose(emb[lt_id], emb[gt_id])))
tag_ids = tok('<c1,CAM_FRONT,100.0,200.0>', add_special_tokens=False).input_ids
speed_ids = tok('speed < 10 km/h', add_special_tokens=False).input_ids
print('tag ids', tag_ids)
print('tag decoded', decode_generated_predictions(tok, [tag_ids])[0])
print('speed ids', speed_ids)
print('speed decoded', decode_generated_predictions(tok, [speed_ids])[0])
print('generation', cfg.generation)
PY

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
mkdir -p "$BASE_PROFILE"
if [ -n "$CACHE_HF_REPO_ID" ] && [ ! -f "$BASE_PROFILE/cache/features.dat" ]; then
  echo "Restoring cache from Hugging Face: $CACHE_HF_REPO_ID"
  hf download "$CACHE_HF_REPO_ID" --repo-type "$CACHE_HF_REPO_TYPE" --local-dir "$BASE_PROFILE" ${CACHE_HF_SUBDIR:+--include "$CACHE_HF_SUBDIR/*"}
else
  echo "Cache restore skipped or cache already exists."
fi
find "$BASE_PROFILE" -maxdepth 3 -type f | head -n 20 || true

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
python -m efficient_vlm_ad prepare-data --config "$RUNTIME_CONFIG" --subset full
if [ -f "$BASE_PROFILE/cache/features.dat" ] && [ -f "$BASE_PROFILE/cache/manifest.json" ]; then
  echo "Feature cache already exists: $BASE_PROFILE/cache"
else
  python -m efficient_vlm_ad prepare-features --config "$RUNTIME_CONFIG" --subset full
fi

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
python -m efficient_vlm_ad diagnose-train --config "$RUNTIME_CONFIG" --stage finetune --debug --debug-samples 1 --debug-numerics


In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$RUNTIME_CONFIG" --stage finetune --max-steps 20 --debug --debug-samples 1 --debug-numerics --progress-log-every-steps 1
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
mkdir -p "$PROFILE/checkpoints"
FINETUNE_TARGET="$PROFILE/checkpoints/$FINETUNE_CHECKPOINT_FILENAME"
if [ -f "$FINETUNE_TARGET" ]; then
  echo "Finetune checkpoint already exists: $FINETUNE_TARGET"
elif [ -n "$FINETUNE_CHECKPOINT_HF_REPO_ID" ]; then
  echo "Restoring finetune checkpoint from Hugging Face: $FINETUNE_CHECKPOINT_HF_REPO_ID/$FINETUNE_CHECKPOINT_FILENAME"
  hf download "$FINETUNE_CHECKPOINT_HF_REPO_ID" "$FINETUNE_CHECKPOINT_FILENAME" --repo-type "$FINETUNE_CHECKPOINT_HF_REPO_TYPE" --local-dir "$PROFILE/checkpoints"
else
  echo "No finetune checkpoint restore requested. Full finetune will start from base model unless finetune_latest.pt already exists locally."
fi
if [ -f "$FINETUNE_TARGET" ]; then
  python -m efficient_vlm_ad verify-resume-checkpoint --config "$RUNTIME_CONFIG" --stage finetune --checkpoint "$FINETUNE_TARGET" --debug --debug-samples 1 --debug-numerics
  cat "$PROFILE/debug/finetune_resume_checkpoint_verify.json"
else
  echo "Skipping finetune resume verify because no finetune checkpoint is present."
fi

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
RESUME=""
if [ -f "$PROFILE/checkpoints/finetune_latest.pt" ]; then
  RESUME="--resume $PROFILE/checkpoints/finetune_latest.pt"
fi
PYTHONUNBUFFERED=1 accelerate launch --multi_gpu --num_processes 2 --num_machines 1 --mixed_precision no --dynamo_backend no -m efficient_vlm_ad train --config "$RUNTIME_CONFIG" --stage finetune $RESUME --debug --debug-samples 1 --debug-numerics
tail -n 20 "$PROFILE/debug/train_progress.jsonl" || true
ls -lh "$PROFILE/checkpoints"

In [ ]:
%%bash
set -e
cd /kaggle/working/Efficient_VLM_For_Autonomous_Driving
. /kaggle/working/router_finetune_only_env.sh
if [ -f "$PROFILE/checkpoints/finetune_best.pt" ]; then
  CHECKPOINT="$PROFILE/checkpoints/finetune_best.pt"
else
  CHECKPOINT="$PROFILE/checkpoints/finetune_latest.pt"
fi
python -m efficient_vlm_ad evaluate --config "$RUNTIME_CONFIG" --checkpoint "$CHECKPOINT" --batch-size 16
python -m efficient_vlm_ad benchmark --config "$RUNTIME_CONFIG" --checkpoint "$CHECKPOINT"
tail -n 20 "$PROFILE/debug/eval_progress.jsonl" || true
tail -n 20 "$PROFILE/debug/benchmark_progress.jsonl" || true
cat "$PROFILE/metrics.json" || true
cat "$PROFILE/generation_diagnostics.json" || true
cat "$PROFILE/benchmark.json" || true
head -n 20 "$PROFILE/predictions.jsonl" || true